In [34]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

import SMI_Methods

In [35]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [36]:
def run_tests_printed(model, tokenizer):
    langs = ["Polish","Czech","Slovak", "Croatian", "Slovene"]
    for lang in tqdm(langs):
        print(lang)
        word_list = prep_words(lang)
        data = prep_data(lang, word_list)
        print("1 Line Data:")
        scores_1_line = [0,0,0,0]
        for i in range(4):
            sc = score_model(model, tokenizer, data[0][i], word_list[i])
            scores_1_line[0] += sc[0]
            scores_1_line[1] += sc[1]
            scores_1_line[2] += sc[2]
            scores_1_line[3] += sc[3]
        print("3 Line Data:")
        scores_3_line = [0,0,0,0]
        for i in range(4):
            sc = score_model(model, tokenizer, data[1][i], word_list[i])
            scores_3_line[0] += sc[0]
            scores_3_line[1] += sc[1]
            scores_3_line[2] += sc[2]
            scores_3_line[3] += sc[3]
        print()
        print("3 Line Data (unfilled):")
        scores_3u_line = [0,0,0,0]
        for i in range(4):
            sc = score_model(model, tokenizer, data[2][i], word_list[i])
            scores_3u_line[0] += sc[0]
            scores_3u_line[1] += sc[1]
            scores_3u_line[2] += sc[2]
            scores_3u_line[3] += sc[3]
        print()
        print("Final values for", lang)
        print("1 line of input")
        print("Top 1 Score:", scores_1_line[0]/4)
        print("Top 3 Score:", scores_1_line[1]/4)
        print("Forced Choiced Combinatorially Optimized Score:", scores_1_line[2]/4)
        print("Forced Choiced Greedy Algorithm Score:", scores_1_line[3]/4)
        print("3 lines of input")
        print("Top 1 Score:", scores_3_line[0]/4)
        print("Top 3 Score:", scores_3_line[1]/4)
        print("Forced Choiced Combinatorially Optimized Score:", scores_3_line[2]/4)
        print("Forced Choiced Greedy Algorithm Score:", scores_3_line[3]/4)
        print("3 lines of input (no fill)")
        print("Top 1 Score:", scores_3u_line[0]/4)
        print("Top 3 Score:", scores_3u_line[1]/4)
        print("Forced Choiced Combinatorially Optimized Score:", scores_3u_line[2]/4)
        print("Forced Choiced Greedy Algorithm Score:", scores_3u_line[3]/4)
        print()

In [37]:
def calculate_scores(model, tokenizer, data, word_list):
    scores = [0, 0, 0, 0]
    for i in range(4):
        sc = SMI_Methods.score_model(model, tokenizer, data[i], word_list[i])
        for j in range(4):
            scores[j] += sc[j]
    return [s / 4 for s in scores]

def run_tests(model, tokenizer):
    langs = ["Polish", "Czech", "Slovak", "Croatian", "Slovene"]
    results = []

    for lang in tqdm(langs):
        word_list = SMI_Methods.prep_words(lang)
        data = SMI_Methods.prep_data(lang, word_list)

        scores_1_line = calculate_scores(model, tokenizer, data[0], word_list)
        scores_3_line = calculate_scores(model, tokenizer, data[1], word_list)
        scores_3u_line = calculate_scores(model, tokenizer, data[2], word_list)

        result_entry = {
            'Language': lang,
            '1 Line - Top 1 Score': scores_1_line[0],
            '1 Line - Top 3 Score': scores_1_line[1],
            '1 Line - FC Combinatorially Optimized Score': scores_1_line[2],
            '1 Line - FC Greedy Algorithm Score': scores_1_line[3],
            '3 Line - Top 1 Score': scores_3_line[0],
            '3 Line - Top 3 Score': scores_3_line[1],
            '3 Line - FC Combinatorially Optimized Score': scores_3_line[2],
            '3 Line - FC Greedy Algorithm Score': scores_3_line[3],
            '3 Line (no fill) - Top 1 Score': scores_3u_line[0],
            '3 Line (no fill) - Top 3 Score': scores_3u_line[1],
            '3 Line (no fill) - FC Combinatorially Optimized Score': scores_3u_line[2],
            '3 Line (no fill) - FC Greedy Algorithm Score': scores_3u_line[3]
        }

        results.append(result_entry)

    df = pd.DataFrame(results)
    return df


In [38]:
#Slovak
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-162M")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-162M").cuda()
df = run_tests(model, tokenizer)
df.to_csv("slovak_162.csv")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [30]:
#Polish
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()
df = run_tests(model, tokenizer)
df.to_csv("polish.csv")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [31]:
#Czech
tokenizer = AutoTokenizer.from_pretrained("spital/gpt2-small-czech-cs")
model = AutoModelForCausalLM.from_pretrained("spital/gpt2-small-czech-cs").cuda()
df = run_tests(model, tokenizer)
df.to_csv("czech.csv")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [32]:
#Slovene
tokenizer = AutoTokenizer.from_pretrained("macedonizer/sl-gpt2")
model = AutoModelForCausalLM.from_pretrained("macedonizer/sl-gpt2").cuda()
df = run_tests(model, tokenizer)
df.to_csv("slovene.csv")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [33]:
#Croatian
tokenizer = AutoTokenizer.from_pretrained("macedonizer/hr-gpt2")
model = AutoModelForCausalLM.from_pretrained("macedonizer/hr-gpt2").cuda()
df = run_tests(model, tokenizer)
df.to_csv("croatian.csv")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
#BERT models, not for results
#Polish
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
run_tests(model, tokenizer)
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
run_tests(model, tokenizer)
#Slovak
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
run_tests(model, tokenizer)
tokenizer = AutoTokenizer.from_pretrained("EMBEDDIA/sloberta")
model = AutoModelForMaskedLM.from_pretrained("EMBEDDIA/sloberta").cuda()
df = run_tests(model, tokenizer)
print(df)

In [ ]:
#Large model
#These are better, but exist in too few languages to experiment
#Do not run, would take a long time

tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
df = run_tests(model, tokenizer)
print(df)

tokenizer = AutoTokenizer.from_pretrained("lchaloupsky/czech-gpt2-oscar")
model = AutoModelForCausalLM.from_pretrained("lchaloupsky/czech-gpt2-oscar").cuda()
df = run_tests(model, tokenizer)
print(df)

In [ ]:
#Serbian
tokenizer = AutoTokenizer.from_pretrained("macedonizer/sr-gpt2")
model = AutoModelForCausalLM.from_pretrained("macedonizer/sr-gpt2").cuda()
df = run_tests(model, tokenizer)
print(df)